# VASP Gemma E2B QLoRA Training (Planner + Refiner)

This notebook trains two adapters on your paired datasets:
- Planner adapter (`planner_e2b_qlora`)
- Refiner adapter (`refiner_e2b_qlora`)

It assumes these files exist in repo:
- `finetune/configs/train_planner_paired_qlora.yaml`
- `finetune/configs/train_refiner_paired_qlora.yaml`
- `finetune/data/planner_paired_train.jsonl`
- `finetune/data/planner_paired_val.jsonl`
- `finetune/data/refiner_paired_train.jsonl`
- `finetune/data/refiner_paired_val.jsonl`


In [1]:
# 1) Check GPU runtime
!nvidia-smi


Mon Apr 20 22:21:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2) Clone repo (edit URL)
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
REPO_DIR = "/content/VASP"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already exists, pulling latest...")
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}


In [ ]:
# 3) Install dependencies
!pip -q install transformers datasets peft accelerate bitsandbytes trl pyyaml huggingface_hub


In [ ]:
# 4) Hugging Face login (needed for google/gemma-4-e2b-it)
from huggingface_hub import login
login()


In [ ]:
# 5) Verify required files
from pathlib import Path
required = [
    "finetune/configs/train_planner_paired_qlora.yaml",
    "finetune/configs/train_refiner_paired_qlora.yaml",
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]
missing = [f for f in required if not Path(f).exists()]
if missing:
    print("Missing files:")
    for m in missing:
        print(" -", m)
    raise SystemExit("Please upload/sync these files before training.")
print("All required files found.")


In [ ]:
# 6) Optional sanity-check dataset counts
import json
from pathlib import Path
for f in [
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]:
    n = sum(1 for _ in Path(f).open("r", encoding="utf-8"))
    print(f"{f}: {n}")


## Train Planner Adapter


In [ ]:
# 7) Train planner
!python -m finetune.training.train_qlora --config finetune/configs/train_planner_paired_qlora.yaml


## Train Refiner Adapter


In [ ]:
# 8) Train refiner
!python -m finetune.training.train_qlora --config finetune/configs/train_refiner_paired_qlora.yaml


In [ ]:
# 9) (Optional) Run evaluation if you have eval config wired to new outputs
# !python -m finetune.evaluation.run_eval --config finetune/configs/eval.yaml


In [ ]:
# 10) Zip trained adapters
!zip -r planner_e2b_qlora.zip finetune/output/planner_e2b_qlora
!zip -r refiner_e2b_qlora.zip finetune/output/refiner_e2b_qlora


In [ ]:
# 11) Download to local machine
from google.colab import files
files.download("planner_e2b_qlora.zip")
files.download("refiner_e2b_qlora.zip")


In [ ]:
# 12) (Optional) Save artifacts to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp planner_e2b_qlora.zip /content/drive/MyDrive/
# !cp refiner_e2b_qlora.zip /content/drive/MyDrive/
